# P0.0 v5 — Test Qwen2.5-32B (4-bit) bang du lieu THAT ma VCB

Henrry 18:00 08/11: thu 32B (co lon tiep theo). Base 4-bit NF4 + double quant, compute float16 — vua T4 free (8-9GB VRAM).
Du lieu that ngay 2026-11-06, nhung thang vao prompt lam context. So sanh chat luong voi ban 1.5B (v2): doc so ty co dung khong, co ket luan truoc khong, co nhac tang von + trich lap Q3 khong.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print(os.path.exists('/content/drive/MyDrive/vn-stock-ai'))
print(os.path.exists('/content/drive/Shared with me/vn-stock-ai'))

In [ ]:
from transformers import AutoTokenizer
tok0 = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-32B-Instruct')
print('tokenizer ok:', tok0.__class__.__name__)

In [ ]:
# Du lieu that VCB (ngay 2026-11-06) — dua vao prompt lam context (item 12)
import json
VCB_DATA = json.loads("""{"ticker": "VCB", "name": "Ngan hang TMCP Ngoai thuong Viet Nam (Vietcombank)", "exchange": "HOSE", "as_of": "2026-11-06", "ohlcv_last20": [{"date": "2026-10-12", "open": 56700.0, "high": 57200.0, "low": 56600.0, "close": 56700.0, "volume": 2068100, "value_ty": 0.12}, {"date": "2026-10-13", "open": 56600.0, "high": 57000.0, "low": 56300.0, "close": 56300.0, "volume": 1581600, "value_ty": 0.09}, {"date": "2026-10-14", "open": 56400.0, "high": 56600.0, "low": 56100.0, "close": 56300.0, "volume": 2155300, "value_ty": 0.12}, {"date": "2026-10-15", "open": 56200.0, "high": 57100.0, "low": 56100.0, "close": 56500.0, "volume": 2368200, "value_ty": 0.13}, {"date": "2026-10-16", "open": 56900.0, "high": 57300.0, "low": 56600.0, "close": 56900.0, "volume": 2388700, "value_ty": 0.14}, {"date": "2026-10-19", "open": 57300.0, "high": 57300.0, "low": 56100.0, "close": 56200.0, "volume": 4165400, "value_ty": 0.24}, {"date": "2026-10-20", "open": 56200.0, "high": 56300.0, "low": 55500.0, "close": 55800.0, "volume": 3671900, "value_ty": 0.2}, {"date": "2026-10-21", "open": 55800.0, "high": 56500.0, "low": 55800.0, "close": 56400.0, "volume": 5480600, "value_ty": 0.31}, {"date": "2026-10-22", "open": 56700.0, "high": 57200.0, "low": 56200.0, "close": 56300.0, "volume": 3290100, "value_ty": 0.19}, {"date": "2026-10-23", "open": 56300.0, "high": 57000.0, "low": 56200.0, "close": 56800.0, "volume": 3278900, "value_ty": 0.19}, {"date": "2026-10-26", "open": 56800.0, "high": 56800.0, "low": 56100.0, "close": 56100.0, "volume": 3617000, "value_ty": 0.2}, {"date": "2026-10-27", "open": 56100.0, "high": 56400.0, "low": 56000.0, "close": 56100.0, "volume": 2438200, "value_ty": 0.14}, {"date": "2026-10-28", "open": 56200.0, "high": 57600.0, "low": 56100.0, "close": 57500.0, "volume": 6453700, "value_ty": 0.37}, {"date": "2026-10-29", "open": 57900.0, "high": 58400.0, "low": 57000.0, "close": 57200.0, "volume": 4049800, "value_ty": 0.23}, {"date": "2026-10-30", "open": 57100.0, "high": 57200.0, "low": 56300.0, "close": 56300.0, "volume": 3621700, "value_ty": 0.21}, {"date": "2026-11-02", "open": 56100.0, "high": 56200.0, "low": 53300.0, "close": 53600.0, "volume": 14009100, "value_ty": 0.76}, {"date": "2026-11-03", "open": 54200.0, "high": 54500.0, "low": 53100.0, "close": 53500.0, "volume": 8910500, "value_ty": 0.48}, {"date": "2026-11-04", "open": 53800.0, "high": 54900.0, "low": 53700.0, "close": 54800.0, "volume": 4883500, "value_ty": 0.27}, {"date": "2026-11-05", "open": 55000.0, "high": 55400.0, "low": 54000.0, "close": 54200.0, "volume": 5003300, "value_ty": 0.27}, {"date": "2026-11-06", "open": 53500.0, "high": 53900.0, "low": 53300.0, "close": 53600.0, "volume": 5202300, "value_ty": 0.28}], "indicators": {"last_close": 53600.0, "change_pct_1d": -1.11, "ma20": 55855, "rsi14": 37.3, "above_ma20": false}, "foreign_last5": [{"date": "6-11-2026", "close": 53600.0, "net_vol": -2065200.0, "net_value_ty": -110.8}, {"date": "5-11-2026", "close": 54200.0, "net_vol": -1628064.0, "net_value_ty": -89.0}, {"date": "4-11-2026", "close": 54800.0, "net_vol": -247411.0, "net_value_ty": -13.5}, {"date": "3-11-2026", "close": 53500.0, "net_vol": -4078253.0, "net_value_ty": -219.1}, {"date": "2-11-2026", "close": 53600.0, "net_vol": -2852652.0, "net_value_ty": -155.1}], "proprietary_last5": [{"date": "6-11-2026", "net_vol": -12500.0, "net_value_ty": -0.7}, {"date": "5-11-2026", "net_vol": 40200.0, "net_value_ty": 2.2}, {"date": "4-11-2026", "net_vol": 102600.0, "net_value_ty": 5.5}, {"date": "3-11-2026", "net_vol": 384500.0, "net_value_ty": 20.6}, {"date": "2-11-2026", "net_vol": 123600.0, "net_value_ty": 7.9}], "news": ["06/11: VCB kiem dinh vung ho tro 53.000-53.500d sau nhip giam manh; phien 02/11 giam ~4,8%, thanh khoan ~14 trieu cp.", "NHNN chap thuan VCB phat hanh rieng le toi da 543,1 trieu cp (6,5% von), von dieu le len ~89.000 ty; Mizuho du kien tham gia.", "9T2026: LNTT ~40.440 ty (+22,1%); Q3 LNTT 11.218 ty di ngang do trich lap du phong 6.184 ty (trong do ~5.000 ty cho 1 khoan trai phieu DN nganh nang luong); NPL 0,71%, LLR 243,2%."], "note": "Gia VND (VNDirect tra x1000). Khoi ngoai VCB dang ban rong 3 phien lien tiep. So lieu that ngay 2026-11-06."}""")

import json
d = VCB_DATA
ind = d["indicators"]
lines = []
lines.append("DU LIEU PHAN TICH MA VCB (ngay %s, HOSE):" % d["as_of"])
lines.append("- Gia dong cua: %s VND (%s%% so voi phien truoc), MA20: %s VND, RSI(14): %s, gia %s MA20."
             % (f"{ind['last_close']:,.0f}", ind["change_pct_1d"],
                f"{ind['ma20']:,.0f}", ind["rsi14"],
                "tren" if ind["above_ma20"] else "duoi"))
lines.append("- Khoi ngoai 5 phien gan nhat (ty VND, am = ban rong): " +
             ", ".join("%s: %s" % (f["date"], f["net_value_ty"]) for f in d["foreign_last5"]) + ".")
lines.append("- Tu doanh 5 phien gan nhat (ty VND): " +
             ", ".join("%s: %s" % (f["date"], f["net_value_ty"]) for f in d["proprietary_last5"]) + ".")
lines.append("- Tin tuc:")
for n in d["news"]:
    lines.append("  + " + n)
lines.append("- Nguon: OHLCV VNDirect, khoi ngoai/tu doanh nmquan.vn, tin tong hop.")
DATA_TEXT = "\n".join(lines)
print(DATA_TEXT)
print("---")
print("So phien OHLCV kem theo:", len(d["ohlcv_last20"]))


In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-32B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-32B-Instruct", quantization_config=bnb, device_map="auto",
    offload_folder="/tmp/offload")
print("model loaded on:", model.device)
print("VRAM used (GB):", torch.cuda.memory_allocated() / 1e9)
# Xem model chia the nao giua GPU/CPU (offload)
_devmap = getattr(model, "hf_device_map", {})
_n_cuda = sum(1 for v in _devmap.values() if str(v).startswith("cuda"))
_n_cpu = sum(1 for v in _devmap.values() if str(v) == "cpu")
print("device map: %d phan tren cuda, %d phan tren cpu" % (_n_cuda, _n_cpu))

SYSTEM = ("Ban la tro ly phan tich chung khoan Viet Nam. Tra loi bang tieng Viet. "
          "Ket luan truoc, roi moi giai thich. Chi dung so lieu duoc cung cap, "
          "khong bia them so. Trinh bay ngan gon co bang neu can. "
          "Cuoi bai them 1 cau mien tru trach nhiem duy nhat.")
USER = DATA_TEXT + "\n\nPhan tich ma VCB: xu huong, dong tien khoi ngoai/tu doanh, vung ho tro/khang cu, va danh gia ngan han."
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": USER},
]
text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok([text], return_tensors="pt").to(model.device)
t0 = time.time()
out = model.generate(**inputs, max_new_tokens=512)
reply = tok.decode(out[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
print("=== REPLY (%.1fs) ===" % (time.time() - t0))
print(reply)


## Ket qua mong doi (de doi chieu voi v2 1.5B)

- Model phai doc dung: gia 53.600 duoi MA20 55.855, RSI(14) 37,3, khoi ngoai BAN RONG 5 phien lien tiep (~-587 ty), tu doanh nho le.
- Vung ho tro 53.000-53.500, cau chuyen tang von (phat hanh rieng le 543,1 tr cp).
- 7B ky vong: doc so ty DUNG don vi (khong thanh %), ket luan truoc, nhac duoc tang von + trich lap Q3, khong cut cau giua chung.
- Ghi lai: VRAM dung bao nhieu GB, thoi gian sinh bao lau, so voi 44,7s cua 1.5B.